# Embedder benchmark on SciFact (BEIR)

## 1 · Runtime check + deps

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo '!! no GPU — Runtime > Change runtime type > T4'
import shutil, sys
print('disk free  :', round(shutil.disk_usage('/').free/2**30, 1), 'GB')
print('ram        : see below')
!free -g | head -2
!pip -q install chromadb sentence-transformers 2>&1 | tail -1
print('\ndeps ready')

name, memory.total [MiB]
Tesla T4, 15360 MiB
disk free  : 65.1 GB
ram        : see below
               total        used        free      shared  buff/cache   available
Mem:              12           1           0           0          11          11

deps ready


## 2 · Fetch the SciFact benchmark (BEIR format)

In [ ]:
!hf download mteb/scifact --repo-type dataset \
    --include "corpus.jsonl" "queries.jsonl" "qrels/test.tsv" \
    --local-dir dataset/scifact

!hf download mteb/scifact corpus.jsonl --repo-type dataset --local-dir /content/dataset/scifact

/usr/local/lib/python3.13/dist-packages/huggingface_hub/cli/download.py:158: UserWarning: Ignoring `--include` since filenames have been explicitly set.
  warnings.warn("Ignoring `--include` since filenames have been explicitly set.")
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files: 100% 2/2 [00:00<00:00, 412.99it/s]
Download complete:           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B                         ✓ Downloaded
  path: /content/dataset/scifact
Download complete:           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
✓ Downloaded
  path: /content/dataset/scifact/corpus.jsonl


## 3 · Install + start Ollama and pull the model

In [ ]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


In [1]:
!ollama pull qwen3-embedding:latest

/bin/bash: line 1: ollama: command not found


In [ ]:
import subprocess, time, os
def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

if sh('command -v ollama').returncode != 0:
    print(sh('curl -fsSL https://ollama.com/install.sh | sh').stdout[-500:])

# start the daemon (no systemd on Colab)
subprocess.Popen(['ollama', 'serve'],
                 stdout=open('/tmp/ollama.log', 'w'),
                 stderr=subprocess.STDOUT)
for _ in range(30):
    if sh('curl -s -m 2 http://localhost:11434/api/version').returncode == 0:
        break
    time.sleep(1)
print(sh('curl -s http://localhost:11434/api/version').stdout)

# trim/extend this list to change the ablation
OLLAMA_MODELS = ['qwen3-embedding:latest']  # the single pick
for m in OLLAMA_MODELS:
    print(f'pulling {m} ...', flush=True)
    r = sh(f'ollama pull {m}')
    assert r.returncode == 0, r.stderr
print(sh('ollama list').stdout)

{"version":"0.40.12"}
pulling qwen3-embedding:latest ...
NAME                      ID              SIZE      MODIFIED               
qwen3-embedding:latest    64b933495768    4.7 GB    Less than a second ago    
qwen3-embedding:4b        df5bd2e3c74c    2.5 GB    3 minutes ago             



## 4 · The embedder

In [ ]:
import requests

class OllamaEF:
    """Embeds via the local Ollama daemon. Per-family task prefixes live here:
    nomic needs search_document:/search_query:, qwen3 is fine plain."""
    DOC_PREFIX = {'*': ''}
    QUERY_PREFIX = {'nomic-embed-text': 'search_query: ', '*': ''}
    def __init__(self, name):
        self.name = name
    def _embed(self, texts, prefix):
        r = requests.post('http://localhost:11434/api/embed', timeout=300,
                          json={'model': self.name, 'input': [prefix + t for t in texts]})
        r.raise_for_status()
        return r.json()['embeddings']
    def embed_documents(self, texts):
        return self._embed(texts, self.DOC_PREFIX.get(self.name, self.DOC_PREFIX['*']))
    def embed_query(self, text):
        return self._embed([text], self.QUERY_PREFIX.get(self.name, self.QUERY_PREFIX['*']))[0]

THE_MODEL = 'qwen3-embedding:0.6b'
def make_embedder(kind, name):
    return OllamaEF(name)
print('embedder ready: OllamaEF →', THE_MODEL)

embedder ready: OllamaEF → qwen3-embedding:0.6b


## 5 · Ingest (chunk 1000/200)

In [ ]:
import json, csv
from chromadb.config import Settings
import chromadb, time

client = chromadb.PersistentClient(path='./chroma_store', settings=Settings(anonymized_telemetry=False))
CHUNK, OVERLAP, BATCH = 1000, 200, 100

def chunk_text(text, size=CHUNK, overlap=OVERLAP):
    # repo-faithful flat windowing (title was prepended upstream)
    step = size - overlap
    return [text[i:i+size] for i in range(0, max(len(text), 1), step)] or ['']

docs = []
for line in open('dataset/scifact/corpus.jsonl'):
    d = json.loads(line)
    full = (d.get('title', '') + '\n\n' + d.get('text', '')).strip()
    for part in chunk_text(full):
        docs.append({'doc_id': str(d['_id']), 'text': part})
print(f'{len(docs)} chunks prepared')

def ingest(kind, model_name):
    col_name = 'scifact_' + model_name.replace('/', '_').replace(':', '_').lower() + '_v1'
    try: client.delete_collection(col_name)
    except Exception: pass
    col = client.get_or_create_collection(col_name)
    ef = make_embedder(kind, model_name)
    t0 = time.time()
    for i in range(0, len(docs), BATCH):
        batch = docs[i:i+BATCH]
        vecs = ef.embed_documents([b['text'] for b in batch])
        col.add(ids=[f'{b["doc_id"]}__{i+j}' for j, b in enumerate(batch)],
                embeddings=vecs,
                metadatas=[{'doc_id': b['doc_id']} for b in batch])
        if (i // BATCH) % 50 == 0:
            print(f'  {i+len(batch)}/{len(docs)}', flush=True)
    print(f'{col_name}: {col.count()} chunks in {time.time()-t0:.0f}s ({col.count()/(time.time()-t0):.0f} ch/s)')
    return col_name

12379 chunks prepared


In [ ]:
SLATE = [('ollama', THE_MODEL)]

In [ ]:
collections = {}
for kind, name in SLATE:
    print(f'### ingesting [{kind}] {name}', flush=True)
    collections[name] = ingest(kind, name)
print('\ndone:', collections)

### ingesting [ollama] qwen3-embedding:0.6b


HTTPError: 404 Client Error: Not Found for url: http://localhost:11434/api/embed

## 6 · Eval

In [ ]:

base = '/content/dataset/scifact'

queries = {d['_id']: d['text']
           for d in (json.loads(l) for l in open(f'{base}/queries.jsonl'))}

qrels = {}
for row in csv.reader(open(f'{base}/qrels/test.tsv'), delimiter='\t'):
    if len(row) >= 3 and row[0] != 'query-id' and int(row[2]) > 0:
        qrels.setdefault(row[0], set()).add(row[1])

covered = [q for q in queries if q in qrels]
print(f'queries: {len(queries)} | qrels qids: {len(qrels)} | covered: {len(covered)}')

import math
def recall_at_k(retrieved, gold, k):
    if not gold: return 0.0
    return len(set(retrieved[:k]) & gold) / len(gold)
def precision_at_k(retrieved, gold, k):
    return len(set(retrieved[:k]) & gold) / k
def mrr_at_k(retrieved, gold, k):
    for i, d in enumerate(retrieved[:k], start=1):
        if d in gold: return 1.0 / i
    return 0.0
def ndcg_at_k(retrieved, gold, k):
    hits = [1.0 if d in gold else 0.0 for d in retrieved[:k]]
    dcg = sum(r / math.log2(i + 2) for i, r in enumerate(hits))
    ideal = [1.0] * min(len(gold), k)
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

HEADROOM = 5   # k*5 chunks fetched -> k distinct docs graded; the load-bearing window
KS = [1, 5, 10]

def evaluate(model_name, kind, col_name):
    col = client.get_collection(col_name)
    ef = make_embedder(kind, model_name)
    accs = {m: {k: [] for k in KS} for m in ('recall', 'precision', 'mrr', 'ndcg')}
    lat = []
    for qi, qid in enumerate(covered):
        gold = qrels[qid]
        t0 = time.perf_counter()
        vec = ef.embed_query(queries[qid])
        res = col.query(query_embeddings=[vec], n_results=max(KS)*HEADROOM,
                        include=['metadatas'])
        lat.append((time.perf_counter()-t0)*1000)
        seen, ordered = set(), []
        for m in res['metadatas'][0]:
            if m['doc_id'] not in seen:
                seen.add(m['doc_id']); ordered.append(m['doc_id'])
        for k in KS:
            accs['recall'][k].append(recall_at_k(ordered, gold, k))
            accs['precision'][k].append(precision_at_k(ordered, gold, k))
            accs['mrr'][k].append(mrr_at_k(ordered, gold, k))
            accs['ndcg'][k].append(ndcg_at_k(ordered, gold, k))
        if (qi+1) % 100 == 0:
            print(f'  {qi+1}/{len(covered)} queries | recall@5='
                  f'{sum(accs["recall"][5])/len(accs["recall"][5]):.3f}', flush=True)
    row = {'embedder': model_name, 'type': kind, 'collection': col_name,
           'num_queries': len(covered)}
    for k in KS:
        for m in accs:
            vals = accs[m][k]
            row[f'{m}@{k}'] = round(sum(vals)/len(vals), 4) if vals else None
    row['latency_ms_mean'] = round(sum(lat)/len(lat), 1) if lat else None
    return row

rows = []
for kind, name in SLATE:
    print(f'### evaluating [{kind}] {name}', flush=True)
    rows.append(evaluate(name, kind, collections[name]))
print('all evaluated')

## 7 · Results

In [ ]:
import pandas as pd
df = pd.DataFrame(rows)
show = ['embedder', 'type', 'recall@1', 'recall@5', 'recall@10', 'ndcg@10', 'mrr@10', 'precision@1', 'latency_ms_mean']
print(df[show].to_string(index=False))

## Reading the results